# Investment Planner — Foundry Skills + Agent Identity

This notebook builds a **prompt agent** on the Foundry managed harness that produces a
6‑month investment plan. Its two heroes are:

1. **Foundry Skills** — reusable, versioned `SKILL.md` + scripts the agent loads on demand.
2. **Agent identity** — every hosted agent runs as its own Microsoft Entra service principal,
   created by the platform at agent creation. Skill scripts in the sandbox authenticate as
   that identity via `DefaultAzureCredential`, so the **agent** (not you) reaches **Key
   Vault**, the user's **Blob Storage**, and the project **Files API** at runtime — regardless
   of where you invoke from. See
   [Hosted agents, part 3](https://ankitbko.github.io/blog/2026/05/hosted-agents-part-3/).

## Scenario

The user drops a holdings CSV in **their own blob container** (same tenant). The agent reads
that blob **and** pulls the user's financial profile from a credential‑protected API — both
via its own identity — analyzes the portfolio in its own sandbox, renders a PDF plan,
and uploads it back to the project. **No key, SAS, or credential ever enters the model's
context.**

## The four skills

| Skill | Role | Identity used |
|---|---|---|
| `financial-profile` | Read Key Vault `sig` → GET profile API → return only profile JSON | agent identity → Key Vault |
| `blob-reader` | Download the holdings CSV from the user's blob (AAD, no key/SAS) | agent identity → Blob Storage |
| `keyvault-secret-reader` | Generic: read any named secret (contrast skill) | agent identity → Key Vault |
| `calling-project-file-api` | Upload the finished plan to the project | agent identity → Files API |

> **The sample owns its tool surface.** `provision_skills.py` (Step 1) creates the toolbox
> (`investment-skills`) and an agent-identity (`AgenticIdentityToken`) project connection (`investment-skills-toolbox`) that
> fronts its MCP endpoint — you do **not** pre-create either, and there is no connection id to
> paste into `.env` (it is derived from `PROJECT_RESOURCE_ID`).

> **Order matters:** the agent identity does not exist until the agent is *created*, so its
> RBAC role assignments happen **after** create (Step 3 below), not before.


## Prerequisites

- An Azure AI Foundry **project** with a deployed model. If you need one, deploy
  [`infrastructure-setup-bicep/40-basic-agent-setup`](../../../../infrastructure/infrastructure-setup-bicep/40-basic-agent-setup).
- **Azure CLI** logged in (`az login`). Your dev identity is used only to *author* (publish
  skills, create the toolbox + connection, create the agent, set the secret, upload the blob)
  — **Azure AI User** on the project, plus rights to set a Key Vault secret and upload a blob.
- A **Key Vault**, a **storage account** (same tenant), and a profile API (a **Logic App**
  returning the fixed profile schema).

### Author-time Azure setup

```bash
# Store ONLY the profile API's shared-access signature (sig) in Key Vault
az keyvault secret set --vault-name <vault> --name financial-profile-sig --value "<the-sig>"

# Upload the holdings CSV to YOUR blob container
az storage blob upload --account-name <account> --container-name <container> \
  --name holdings.csv --file ./data/holdings.csv --auth-mode login
```

The **runtime** role assignments (to the *agent* identity) come later, in Step 3 — after the
agent exists. Copy `.env.sample` to `.env` and fill in the values (including
`PROJECT_RESOURCE_ID`) before running the cells.


In [ ]:
import os
import dotenv

from sample_config import AGENT_NAME, TOOLBOX_CONNECTION_NAME, toolbox_mcp_url

dotenv.load_dotenv()

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"].rstrip("/")
MODEL = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]
PROJECT_RESOURCE_ID = os.environ["PROJECT_RESOURCE_ID"]
HOLDINGS_BLOB_URL = os.environ["HOLDINGS_BLOB_URL"]
KEYVAULT_URL = os.environ["KEYVAULT_URL"]   
FINANCIAL_PROFILE_URL = os.environ["FINANCIAL_PROFILE_URL"]


# The toolbox (investment-skills) and its connection are created by provision_skills.py (Step 1).
# The MCP tool points at the toolbox's data-plane MCP endpoint; the connection (referenced by
# name) supplies AgenticIdentityToken auth.
TOOLBOX_MCP_URL = f"{toolbox_mcp_url(PROJECT_ENDPOINT)}?api-version=v1"
print("Holdings Blob URL:", HOLDINGS_BLOB_URL)
print("Financial Profile URL:", FINANCIAL_PROFILE_URL)
print("KeyVault URL:", KEYVAULT_URL)
print("Project:   ", PROJECT_ENDPOINT)
print("Model:     ", MODEL)
print("Toolbox:   ", TOOLBOX_MCP_URL)
print("Connection:", TOOLBOX_CONNECTION_NAME)
print("Blob:      ", os.environ.get("HOLDINGS_BLOB_URL", "(set HOLDINGS_BLOB_URL)"))

## Step 1 — Publish the skills, toolbox, and connection (run once)

The installed `azure-ai-projects` preview does not expose `beta.skills`, so publishing uses
the data‑plane **Skills REST API**, wrapped in `provision_skills.py`. It packages every file
under each `skills/<name>/` folder (including `scripts/`), publishes the four skills, then
**creates the toolbox** (`investment-skills`) with those skills as its default version and
**creates the agent-identity project connection** (`investment-skills-toolbox`) that fronts the
toolbox's MCP endpoint. Re‑run whenever you edit a skill.


In [ ]:
import subprocess, sys

result = subprocess.run([sys.executable, "provision_skills.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit("provision_skills.py failed — fix the error above before continuing.")

## Step 2 — Create the prompt agent

The agent gets two tools: **`CodeInterpreterTool`** (the built‑in "Hand" that runs the skills'
scripts under the agent identity) and **`MCPTool`** (references the toolbox via the connection
created in Step 1). Creating the agent is what **materializes the agent identity** whose
principal id we grant roles to in Step 3.


In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import CodeInterpreterTool, MCPTool, PromptAgentDefinition
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())
openai_client = project_client.get_openai_client()

INSTRUCTIONS = """\
You are an investment-planning assistant that produces a 6-month allocation plan for the
user's portfolio. You are driven turn by turn: do ONLY what the current turn asks and do
not front-run later steps. In particular, never upload a file or construct a project
endpoint until a turn explicitly asks you to. Use the attached skills and never invent data.

Skills and guardrails:
- `financial-profile`: reads its API credential from Key Vault using your managed identity —
  never ask the user for it and never print the credential.
- `blob-reader`: downloads the holdings CSV from the user's Azure Blob Storage (env
  HOLDINGS_BLOB_URL) using your managed identity — never ask for a key or SAS. Parse the CSV
  in your sandbox with plain Python (columns: ticker, name, sector, qty, avg_cost_usd,
  current_price_usd, dividend_yield_pct, beta, analyst_rating).
- `calling-project-file-api`: uploads a file to the project and returns a file id. Use it
  ONLY when a turn explicitly asks you to upload.

When a turn asks you to build the allocation, respect the profile's risk_tolerance,
investable_cash_usd, goals, and constraints (e.g. honor `no_crypto`); show target weights,
specific buy/trim actions for the investable cash, and a one-line rationale per action.
Render the plan as a PDF with reportlab and make the last line of the PDF exactly:
"This is a generated example and not financial advice."
"""

definition = PromptAgentDefinition(
    model=MODEL,
    instructions=INSTRUCTIONS,
    temperature=0,
    tools=[
        MCPTool(
            server_url=TOOLBOX_MCP_URL,
            server_label="toolbox",
            require_approval="never",
            project_connection_id=TOOLBOX_CONNECTION_NAME,
        ),
    ],
)
# Run in the managed harness so skills execute server-side under the agent identity.
definition["harness"] = "ghcp"

agent = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=definition,
)
print(f"Agent created (id={agent.id}, name={agent.name}, version={agent.version})")

## Step 3 — Grant the agent identity its runtime roles

The agent now has an Entra service principal. Fetch its `instance_identity.principal_id` and
assign the roles it needs to read Key Vault, read the blob, and call the Files API. **Run the
printed `az` commands** (or wire in your own automation), then wait ~1–5 minutes for RBAC to
propagate before Step 4.


In [ ]:
# Fetch the agent identity's principal id (only available after the agent exists).
pid = subprocess.run(
    ["az", "rest", "--method", "GET",
     "--url", f"{PROJECT_ENDPOINT}/agents/{AGENT_NAME}?api-version=v1",
     "--resource", "https://ai.azure.com",
     "--query", "instance_identity.principal_id", "-o", "tsv"],
    capture_output=True, text=True, shell=(os.name == "nt"),
).stdout.strip()
print("Agent identity principal id:", pid or "(not found — check az login / agent name)")

grants = [
    ("Key Vault Secrets User", os.environ.get("KEYVAULT_RESOURCE_ID", "<key-vault-resource-id>")),
    ("Storage Blob Data Reader", os.environ.get("STORAGE_RESOURCE_ID", "<storage-account-resource-id>")),
    ("Foundry User", os.environ.get("PROJECT_RESOURCE_ID", "<foundry-project-resource-id>")),  # Files API
]
print("\nRun these (fill scopes from .env if you left placeholders):\n")
for role, scope in grants:
    print(f"az role assignment create --assignee-object-id {pid} "
          f"--assignee-principal-type ServicePrincipal --role \"{role}\" --scope {scope}\n")

## Step 4 — Run the agent

After the roles have propagated, invoke the agent. It orchestrates: pull profile
(agent identity → Key Vault) → read holdings blob (agent identity → Blob Storage) → analyze →
write plan → upload (agent identity → Files API) → summarize. A 401/403 from a skill here
means the matching role hasn't propagated yet — wait and retry.


In [ ]:
import time
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())
openai_client = project_client.get_openai_client()

# One conversation shared by every turn: this is what keeps all turns on the SAME
# warm session/sandbox, so files written in one turn persist to later turns and the
# x-agent-session-id stays stable. A single mega-prompt tends to time out (long turn)
# and can hit mid-turn token expiry, so we split the work into short sequential turns.
conversation = openai_client.conversations.create()
print(f"Conversation created: {conversation.id}\n")

def stream_turn(prompt):
    """Send one turn and stream its output/tool/reasoning events as they arrive."""
    all_events = []
    start = time.time()
    session_id = None
    with openai_client.responses.with_streaming_response.create(
        conversation=conversation.id,
        model=MODEL,
        input=prompt,
        stream=True,
        extra_body={"agent_reference": {"type": "agent_reference", "name": AGENT_NAME}},
    ) as api_response:
        session_id = api_response.headers.get("x-agent-session-id")
        print(f"HTTP status: {api_response.status_code}")
        print(f"x-request-id: {api_response.headers.get('x-request-id')}")
        print(f"x-agent-session-id: {session_id}")
        print("\n" + "-" * 80 + "\n")
        for event in api_response.parse():
            all_events.append(event.type)
            if event.type == "response.created":
                print(f"\U0001f680 Response ID: {event.response.id}\n")
            elif event.type == "response.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "response.output_item.added":
                item = getattr(event, "item", None)
                if item is not None:
                    if item.type == "function_call":
                        print(f"\n\n\U0001f527 Tool call: {item.name}")
                    elif item.type == "reasoning":
                        print("\n\U0001f9e0 Reasoning...")
                    else:
                        print(f"\n\U0001f4e6 Output item: {item.type}")
            elif event.type == "response.reasoning.delta":
                if hasattr(event, "delta"):
                    print(event.delta, end="", flush=True)
            elif event.type == "response.function_call_arguments.done":
                if hasattr(event, "arguments"):
                    print(f"\n    args: {event.arguments[:200]}")
            elif event.type == "response.output_item.done":
                item = getattr(event, "item", None)
                if item is not None and item.type == "function_call":
                    print(f"    \u2705 Tool call done: {item.name}")
                elif item is not None and item.type == "function_call_output":
                    print(f"    \U0001f4cb Tool output: {getattr(item, 'output', '')[:300]}")
            elif event.type == "response.output_text.done":
                print()
            elif event.type == "response.completed":
                print("\n\n\u2705 Response completed")
            elif event.type == "response.failed":
                print("\n\n\u274c Response FAILED")
                err = getattr(getattr(event, "response", None), "error", None)
                if err is not None:
                    print(f"    Error: {err}")
            elif event.type == "response.incomplete":
                print("\n\n\u26a0\ufe0f Response incomplete")
            elif event.type == "response.reasoning_summary_text.delta":
                if hasattr(event, "delta"):
                    print(event.delta, end="", flush=True)
    print(f"\n{'=' * 80}")
    print(f"Turn time: {time.time() - start:.1f}s | events: {len(all_events)}")
    return session_id


# Short, sequential turns in the same conversation. Each depends on prior turns'
# on-disk output (same sandbox). Relevant URLs are injected only where needed.
PLAN_FILE_PATH = "/workspace/data/six_month_plan.pdf"

turns = [
    # 1) Optional setup. Drop this turn if the sandbox already has the SDKs and you
    #    never see ModuleNotFoundError. Skills are meant to be self-contained.
    "Set up the environment: make sure azure-identity, azure-storage-blob, "
    "azure-keyvault-secrets and reportlab are importable, installing any that are "
    "missing. Reply READY when done.",
    # 2) Download the holdings CSV via the blob-reader skill (agent identity, no key/SAS).
    "Use the blob-reader skill to download my holdings CSV from my blob storage using "
    f"your managed identity. Holdings blob URL: {HOLDINGS_BLOB_URL}. "
    "Confirm the local file path and the row count.",
    # 3) Fetch the financial profile via the financial-profile skill (Key Vault sig -> API).
    "Use the financial-profile skill to fetch my profile JSON (risk_tolerance, "
    "investable_cash_usd, horizon_months, goals, constraints). "
    f"Key Vault URL: {KEYVAULT_URL}. Profile API URL: {FINANCIAL_PROFILE_URL}. "
    "Summarize the profile — never print the credential.",
    # 4) Analyze on the Hand (same sandbox/disk as turns 2 & 3) and RENDER the plan to a
    #    PDF on disk. Do NOT use the code interpreter tool — it's a separate sandbox and
    #    would not see the downloaded CSV/profile, and its output would not be on the Hand
    #    disk for turn 5 to upload. Render the PDF directly with reportlab (pure-Python).
    "Using plain Python in your sandbox (the same environment where the CSV and profile "
    "were downloaded), parse the downloaded CSV, compute position values and portfolio "
    "weights, then build a 6-month allocation that respects my risk_tolerance, "
    "investable_cash_usd and constraints. Render the result as a PDF using reportlab "
    "(title, profile summary, holdings table, and the 6-month allocation table) and write "
    f"it to {PLAN_FILE_PATH}. Confirm the file exists and report its byte size. Do ONLY "
    "this now — do not upload, do not construct any project endpoint, and do not create "
    "any other file; stop after reporting the byte size.",
    # 5) Upload the already-produced PDF via the calling-project-file-api skill (pure upload).
    "Use the calling-project-file-api skill to upload the PDF already on disk at "
    f"{PLAN_FILE_PATH} to the project ({PROJECT_ENDPOINT}) with content type "
    "application/pdf. Return the uploaded file id.",
]

session_ids = []
for i, turn in enumerate(turns, 1):
    print("\n" + "#" * 80)
    print(f"# TURN {i}/{len(turns)}: {turn[:70]}...")
    print("#" * 80 + "\n")
    session_ids.append(stream_turn(turn))
    time.sleep(3)  # let the Brain mark the turn complete before the next

# All turns should share ONE session id; a change means a cold pod dropped sandbox
# state (re-run from turn 2 if so).
unique_sessions = sorted(set(s for s in session_ids if s))
print("\n" + "=" * 80)
print(f"Session id(s) across turns: {unique_sessions}")
if len(unique_sessions) > 1:
    print("\u26a0\ufe0f  Session id changed mid-run \u2014 sandbox state may have been lost.")


In [ ]:
# Download the uploaded plan and save it locally as a .pdf
FILE_ID = "assistant-2XxgiJERC85DmZRwkW5jbE"   # id returned by turn 5
LOCAL_PATH = "six_month_plan.pdf"

# openai_client comes from earlier in the notebook:
#   project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())
#   openai_client  = project_client.get_openai_client()

content = openai_client.files.content(FILE_ID)   # binary content of the file

# The SDK's binary response supports write_to_file; fall back to raw bytes if not.
if hasattr(content, "write_to_file"):
    content.write_to_file(LOCAL_PATH)
else:
    data = content.read() if hasattr(content, "read") else content.content
    with open(LOCAL_PATH, "wb") as f:
        f.write(data)

import os
print(f"Saved {LOCAL_PATH} ({os.path.getsize(LOCAL_PATH)} bytes)")

## Step 5 — Clean up

Delete the agent version. (Published skills, the toolbox, the connection, and the uploaded
plan file remain in the project.)


In [ ]:
project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
project_client.close()
print("Agent version deleted.")

## Why this order (and secret hygiene)

**Agent identity is created with the agent**, so its RBAC role assignments can only happen
*after* Step 2 — assigning them earlier is impossible because the service principal doesn't
exist yet. At runtime the platform delivers the *agent's* token to `DefaultAzureCredential`
inside the sandbox, so the agent identity is what reads Key Vault / Blob Storage / Files API
no matter where you invoke from. Your local `az login` identity only authored the setup.

**Native toolbox, no stored key.** The toolbox lives in the same project, so its connection
uses the **`AgenticIdentityToken`** auth type — the agent's own identity token (its `Azure AI User` role) authorizes the MCP call.
No API key or SAS is stored on the connection.

**Secret hygiene:** only the profile API's `sig` lives in Key Vault; the base URL is
non‑secret config, and blob access uses **AAD** (no key/SAS to leak). The `financial-profile`
skill reconstructs `url + "&sig=" + sig` **inside the script process** and returns only the
profile JSON — the credential never enters a prompt, a tool argument, or the model's context.
Contrast the generic `keyvault-secret-reader`, which *can* surface the raw value: prefer
composed skills whenever the secret is only a means to an end.

> This is a generated example and not financial advice.
